# QM640 · 07b API instruments: I7 Gemini Flash, DeepSeek, and the Claude Opus 5.5 frontier benchmark
CPU runtime (no GPU units). Can run at the same time as notebooks 06 or 07.

**Secrets (key icon on the left, notebook access on):**
- `GEMINI_API_KEY`: free key from Google AI Studio (I7).
- `FRONTIER_GATEWAY_URL` and `FRONTIER_GATEWAY_TOKEN`: the Replit gateway, which calls Claude Opus 5.5 (Anthropic) and DeepSeek on Replit credits.

An instrument whose secret is missing is skipped, so this notebook can run with Gemini alone first. Tickets are sent in batches (8 per call for classification, 4 for search) with JSON answers. The free Gemini tier has a daily limit: if it is reached, run all again the next day and finished batches are kept.

Training eligibility (decision record D-7): DeepSeek yes (MIT); Gemini and Claude Opus 5.5 no (benchmark only).

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
from pathlib import Path
DATA_ROOT = Path('/content/drive/MyDrive/Walsh DBA/Doctoral Research/QM640/Data')
def secret(name):
    try: return userdata.get(name)
    except Exception: return None
GEMINI_KEY = secret('GEMINI_API_KEY')               # free key from Google AI Studio
GATEWAY_URL = secret('FRONTIER_GATEWAY_URL')        # Replit gateway (frontier benchmark and DeepSeek on Replit credits)
GATEWAY_TOKEN = secret('FRONTIER_GATEWAY_TOKEN')
GEMINI_MODEL = None                  # None = newest Flash model available to the key
FRONTIER_MODEL = 'claude-opus-5-5'   # Claude Opus 5.5 (Anthropic): frontier accuracy benchmark only, traces not training-eligible
DEEPSEEK_MODEL = 'deepseek/deepseek-chat'   # OpenRouter id; check the gateway's model list
GEMINI_PRICE = (0.30, 2.50)          # US$ per million input/output tokens at paid-tier list price (free tier costs nothing); check current price
FRONTIER_PRICE = (4.0, 20.0)         # Claude Opus 5.5 list price
DEEPSEEK_PRICE = (0.30, 1.20)        # check OpenRouter's current price
GEMINI_MIN_GAP = 7                   # seconds between Gemini calls (free tier allows about 10 per minute)
CLS_BATCH, RET_BATCH = 8, 4          # tickets per call
RUN = ['I7', 'FRONTIER', 'DEEPSEEK']         # instruments for RQ1
RUN_CONTEXT = ['I7', 'FRONTIER', 'DEEPSEEK'] # instruments for the RQ3 context runs
try:
    import google.genai
except ImportError:
    !pip -q install google-genai

## 1. Settings and batch prompts
Same tickets, label lists and question wording as the other instruments; several tickets per call to save cost and stay within the free-tier limits.

In [ ]:
import json, time, re, random, difflib, datetime, numpy as np, pandas as pd, requests
from sklearn.metrics import f1_score
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
JIRA = DATA_ROOT/'processed'/'jira'; TESTS = DATA_ROOT/'processed'/'jira_tests'; RESULTS = DATA_ROOT/'results'
RAW = RESULTS/'api_raw'; RAW.mkdir(parents=True, exist_ok=True); FILES = sorted(JIRA.glob('*.parquet')); SEED = 640
LABELS = json.load(open(TESTS/'label_spaces.json'))
CLS = {'D1_issuetype': ('issuetype', 'type', 'What type of issue is this?'), 'D1_priority': ('priority', 'priority', 'What priority should this issue be given?'),
       'D4_resolution': ('resolution', 'resolution', 'How is this issue most likely to be resolved?'),
       'D5_component': ('proj_comp', 'component', 'Which project component (team) should handle this issue?')}
RET = {'D2_duplicate': 'Which of the earlier issues listed does this new issue duplicate?',
       'D6_cause': 'Which of the earlier issues listed caused the problem reported in this new issue?'}
def txt(x): return x if isinstance(x, str) else ''
def first_comp(s):
    try: c = json.loads(s)
    except Exception: return None
    return c[0] if c else None
def ticket(r, n=1200): return f"Project: {r['project']}\nSummary: {txt(r['summary'])}\nDescription: {txt(r['description'])[:n]}"
def cls_batch_prompt(items, tasks, contexts=None):
    qs = '\n\n'.join(f"{CLS[t][1]}: {CLS[t][2]}\nOptions: " + ' ; '.join(LABELS[t]) for t in tasks)
    tk = '\n\n'.join(f"[{i}]\n{ticket(r)}" + (f"\nSimilar earlier issues and how they were handled:\n{contexts[i]}" if contexts and contexts.get(i) else '') for i, r in items)
    fields = ', '.join(f'"{CLS[t][1]}": "<option>"' for t in tasks)
    return (f"You are triaging issues in a software issue tracker. For every ticket, answer each question with exactly one option from its list, copied exactly.\n\n"
            f"Questions:\n{qs}\n\nTickets:\n{tk}\n\n"
            f'Reply with JSON only: {{"answers": [{{"id": "<ticket id>", {fields}, "confidence": <0 to 1, how sure you are overall>}}]}}')
def ret_batch_prompt(task, items, texts):
    blocks = []
    for i, (r, cands) in items:
        lines = '\n'.join(f"  c{j+1}: {txt(texts.loc[c, 'summary'])} | {txt(texts.loc[c, 'description'])[:200]}" for j, c in enumerate(cands) if c in texts.index)
        blocks.append(f"[{i}] New issue:\n{ticket(r, 800)}\nEarlier issues:\n{lines}\n  none: None of the listed issues")
    return (f"{RET[task]} Answer for each new issue separately.\n\n" + '\n\n'.join(blocks) +
            '\n\nReply with JSON only: {"answers": [{"id": "<issue id>", "choice": "<c1..c10 or none>", "confidence": <0 to 1>}]}')
def to_label(text, space):
    t = (text or '').strip().strip('.').strip('"')
    low = {o.lower(): o for o in space}
    if t.lower() in low: return low[t.lower()]
    m = difflib.get_close_matches(t, space, n=1, cutoff=0.6)
    return m[0] if m else 'Other'
def parse_json(text):
    m = re.search(r'\{.*\}', text or '', re.S)
    try: return json.loads(m.group(0))['answers'] if m else []
    except Exception: return []
tick = pd.read_parquet(TESTS/'jira_test_tickets.parquet').drop_duplicates('key').set_index('key', drop=False)
tests = {t: pd.read_csv(TESTS/f'testset_jira_{t}.csv') for t in list(CLS) + list(RET)}
HAS_CONTEXT = (TESTS/'context_i2_i3.csv').exists()
print(len(tick), 'test tickets; context from notebook 05:', HAS_CONTEXT)

## 2. Instrument clients
Gemini is called directly; Claude Opus 5.5 and DeepSeek go through the Replit gateway, billed to Replit credits at public API prices.

In [ ]:
class Client:
    def __init__(self, name, call, min_gap, price_in, price_out, training_eligible):
        self.name, self.call, self.min_gap, self.pin, self.pout, self.eligible = name, call, min_gap, price_in, price_out, training_eligible
        self.last, self.stop = 0.0, None
    def ask(self, prompt):
        for attempt in range(8):
            wait = self.min_gap - (time.time() - self.last)
            if wait > 0: time.sleep(wait)
            self.last = time.time()
            try:
                return self.call(prompt)
            except Exception as e:
                msg = str(e)
                if 'per day' in msg.lower() or 'PerDay' in msg:
                    self.stop = 'daily quota reached: run this notebook again tomorrow; finished batches are kept'; return None
                if any(c in msg for c in ('429', '500', '502', '503', '504', 'RESOURCE_EXHAUSTED', 'timed out', 'Timeout')):
                    time.sleep(min(120, 5 * 2**attempt)); continue
                print(self.name, 'error:', msg[:300]); return None
        return None
def gemini_call(prompt):
    r = gclient.models.generate_content(model=GEMINI_MODEL, contents=prompt,
                                        config={'temperature': 0, 'response_mime_type': 'application/json'})
    u = r.usage_metadata
    return r.text, (u.prompt_token_count or 0), (u.candidates_token_count or 0) + (getattr(u, 'thoughts_token_count', 0) or 0)
def gateway_call(model, provider):
    def call(prompt):
        r = requests.post(GATEWAY_URL.rstrip('/') + '/v1/complete', timeout=300, headers={'Authorization': f'Bearer {GATEWAY_TOKEN}'},
                          json={'provider': provider, 'model': model, 'prompt': prompt, 'max_tokens': 4000, 'temperature': 0})
        if r.status_code != 200: raise RuntimeError(f'{r.status_code} {r.text[:200]}')
        o = r.json(); return o['text'], o['usage']['input_tokens'], o['usage']['output_tokens']
    return call
clients = {}
if GEMINI_KEY:
    from google import genai
    gclient = genai.Client(api_key=GEMINI_KEY)
    if not GEMINI_MODEL:
        names = [m.name.split('/')[-1] for m in gclient.models.list()]
        flash = sorted(n for n in names if 'flash' in n and not any(x in n for x in ('lite', 'image', 'tts', 'live', 'audio', 'exp', 'preview')))
        GEMINI_MODEL = flash[-1]; print('Gemini Flash models found:', flash)
    clients['I7'] = Client(f'I7 {GEMINI_MODEL}', gemini_call, GEMINI_MIN_GAP, *GEMINI_PRICE, False)
if GATEWAY_URL:
    clients['FRONTIER'] = Client(f'Frontier benchmark {FRONTIER_MODEL}', gateway_call(FRONTIER_MODEL, 'anthropic'), 0.5, *FRONTIER_PRICE, False)
    clients['DEEPSEEK'] = Client(f'DeepSeek {DEEPSEEK_MODEL}', gateway_call(DEEPSEEK_MODEL, 'openrouter'), 0.5, *DEEPSEEK_PRICE, True)
print('instruments ready:', {k: c.name for k, c in clients.items()})

## 3. Run helper (batched, resumable; logs tokens and US$ per batch)

In [ ]:
def run(inst, name, batches):
    """batches: list of (batch_id, [ids], prompt). One JSON line per batch; reruns skip finished batches."""
    c = clients[inst]; f = RAW/f'{inst.lower()}_{name}.jsonl'; done = set()
    if f.exists():
        for l in open(f): done.add(json.loads(l)['batch_id'])
    todo = [b for b in batches if b[0] not in done]; t0 = time.time()
    for n, (bid, ids, prompt) in enumerate(todo, 1):
        if c.stop: break
        out = c.ask(prompt)
        if out is None: continue
        text, tin, tout = out
        rec = {'batch_id': bid, 'ids': ids, 'text': text, 'input_tokens': tin, 'output_tokens': tout,
               'usd': tin/1e6*c.pin + tout/1e6*c.pout, 'seconds': time.time() - t0, 'instrument': c.name, 'training_eligible': c.eligible}
        with open(f, 'a') as fh: fh.write(json.dumps(rec) + '\n')
        if n % 20 == 0: print(f'  {name}: {n}/{len(todo)} batches')
    print(f'{inst} {name}: {len(todo)} batches attempted', '| STOPPED: ' + c.stop if c.stop else '')
def batches_cls(keys, tasks, size, contexts=None):
    out = []
    for b in range(0, len(keys), size):
        ks = keys[b:b+size]; items = [(k, tick.loc[k]) for k in ks]
        out.append((f'{tasks[0] if len(tasks) == 1 else "all"}-{b}', ks, cls_batch_prompt(items, tasks, contexts)))
    return out
need = set()
for t in RET: need |= {c for s in pd.read_csv(TESTS/f'i1_predictions_{t}.csv')['i1_top10'] for c in json.loads(s)}
texts = pd.concat([tick[['key', 'summary', 'description']]] + [pd.read_parquet(f, columns=['key', 'summary', 'description'], filters=[('key', 'in', list(need - set(tick.index)))]) for f in FILES]).drop_duplicates('key').set_index('key')
shortlist = {t: dict(zip(*pd.read_csv(TESTS/f'i1_predictions_{t}.csv')[['key', 'i1_top10']].values.T)) for t in RET}
def batches_ret(t, sl, size):
    keys = [k for k in tests[t]['key'] if isinstance(sl.get(k), str)]
    return [(f'{t}-{b}', keys[b:b+size], ret_batch_prompt(t, [(k, (tick.loc[k], json.loads(sl[k]))) for k in keys[b:b+size]], texts)) for b in range(0, len(keys), size)]
cls_keys = sorted(set().union(*[set(tests[t]['key']) for t in CLS]))
print(len(cls_keys), 'classification tickets;', len(batches_cls(cls_keys, list(CLS), CLS_BATCH)), 'batches per instrument')

## 4. RQ1: classification and search test sets

In [ ]:
for inst in RUN:
    if inst not in clients: continue
    run(inst, 'classification', batches_cls(cls_keys, list(CLS), CLS_BATCH))
    for t in RET: run(inst, t, batches_ret(t, shortlist[t], RET_BATCH))

## 5. RQ3: with company history (needs notebook 05)

In [ ]:
if HAS_CONTEXT:
    ctx = pd.read_csv(TESTS/'context_i2_i3.csv')
    ctick = pd.read_parquet(TESTS/'context_tickets.parquet').drop_duplicates('key').set_index('key')
    ctick['proj_comp'] = [f'{p} | {c}' if c else None for p, c in zip(ctick['project'], ctick['components'].map(first_comp))]
    sub = ctx[ctx['decision'] == 'D5_component'].set_index('key')['i2_top10']
    contexts = {k: '\n'.join(f"- {txt(ctick.loc[c, 'summary'])[:150]} -> component: {ctick.loc[c, 'proj_comp']}" for c in json.loads(v)[:5] if c in ctick.index) for k, v in sub.items()}
    hyb = {t: dict(zip(ctx[ctx['decision'] == t]['key'], ctx[ctx['decision'] == t]['hybrid_top10'])) for t in RET}
    extra = {c for t in RET for s in hyb[t].values() if isinstance(s, str) for c in json.loads(s)} - set(texts.index)
    texts = pd.concat([texts.reset_index()] + [pd.read_parquet(f, columns=['key', 'summary', 'description'], filters=[('key', 'in', list(extra))]) for f in FILES]).drop_duplicates('key').set_index('key')
    for inst in RUN_CONTEXT:
        if inst not in clients: continue
        run(inst, 'ctx_D5_component', batches_cls(list(tests['D5_component']['key']), ['D5_component'], CLS_BATCH, contexts))
        for t in RET: run(inst, f'ctx_{t}', batches_ret(t, hyb[t], RET_BATCH))

## 6. Results: classification

In [ ]:
def wilson(k, n): lo, hi = proportion_confint(k, n, method='wilson'); return f'{100*lo:.1f}–{100*hi:.1f}'
def coverage(conf, ok, p):
    o = np.argsort(-conf); prec = np.cumsum(ok[o])/np.arange(1, len(o)+1); idx = np.where(prec >= p)[0]
    return round(100*(idx.max()+1)/len(o), 1) if len(idx) else 0.0
def ece(conf, ok, m=10):
    e = 0
    for a, b in zip(np.linspace(0, 1, m+1)[:-1], np.linspace(0, 1, m+1)[1:]):
        s = (conf > a) & (conf <= b)
        if s.any(): e += s.mean()*abs(ok[s].mean() - conf[s].mean())
    return round(e, 3)
def load(inst, name):
    f = RAW/f'{inst.lower()}_{name}.jsonl'; ans, cost, n = {}, 0.0, 0
    if not f.exists(): return ans, np.nan
    for l in open(f):
        o = json.loads(l); cost += o['usd']; n += len(o['ids'])
        for a in parse_json(o['text']):
            if isinstance(a, dict) and 'id' in a: ans[str(a['id'])] = a
    return ans, 1000*cost/max(n, 1)
i1 = {t: dict(zip(*pd.read_csv(TESTS/f'i1_predictions_{t}.csv')[['key', 'i1_pred']].values.T)) for t in CLS}
rows, preds = [], []
for inst in clients:
    for name, tasks, label in (('classification', list(CLS), ''), ('ctx_D5_component', ['D5_component'], ' + I2 context')):
        ans, usd_k = load(inst, name)
        if not ans: continue
        for t in tasks:
            ts = tests[t]; y = ts['label'].values
            p = np.array([to_label(ans.get(k, {}).get(CLS[t][1]), LABELS[t]) for k in ts['key']])
            conf = np.array([float(ans.get(k, {}).get('confidence', 0) or 0) for k in ts['key']])
            ok = p == y; ok1 = np.array([i1[t].get(k) == lab for k, lab in zip(ts['key'], y)]); b, c = int((ok1 & ~ok).sum()), int((~ok1 & ok).sum())
            rows.append({'Decision': t, 'Instrument': clients[inst].name + label, 'Test rows': len(y), 'Answered %': round(100*np.mean([k in ans for k in ts['key']]), 1),
                         'Accuracy %': round(100*ok.mean(), 1), '95% CI (Wilson)': wilson(int(ok.sum()), len(y)), 'Macro-F1': round(f1_score(y, p, average='macro', zero_division=0), 3),
                         'I1 accuracy %': round(100*ok1.mean(), 1), 'McNemar p (vs I1)': round(mcnemar([[0, b], [c, 0]], exact=True).pvalue, 4),
                         'Coverage % at 90% precision (stated confidence)': coverage(conf, ok, .90), 'ECE (stated confidence)': ece(conf, ok),
                         'Cost US$ per 1,000 tickets': round(usd_k, 3), 'Training-eligible traces': clients[inst].eligible})
            preds.append(pd.DataFrame({'instrument': clients[inst].name + label, 'decision': t, 'key': ts['key'], 'label': y, 'pred': p, 'conf': conf}))
api = pd.DataFrame(rows); api.to_csv(RESULTS/'rq1_jira_api_instruments.csv', index=False)
if preds: pd.concat(preds).to_csv(TESTS/'api_predictions_classification.csv', index=False)
api

## 7. Results: duplicate and cause search

In [ ]:
rrows = []
for inst in clients:
    for pre, sl_name in (('', 'I1 shortlist'), ('ctx_', 'I1+I2 hybrid shortlist')):
        for t in RET:
            ans, usd_k = load(inst, pre + t)
            if not ans: continue
            sl = shortlist[t] if not pre else hyb[t]; hit = n = 0
            for k, truth in zip(tests[t]['key'], tests[t]['answers']):
                if not isinstance(sl.get(k), str): continue
                n += 1; cands = json.loads(sl[k]); ch = str(ans.get(k, {}).get('choice', '')).lower()
                idx = int(re.sub(r'\D', '', ch) or 0) if ch.startswith('c') else 0
                hit += bool(0 < idx <= len(cands) and cands[idx-1] in set(json.loads(truth)))
            rrows.append({'Decision': t, 'Instrument': f'{clients[inst].name} re-rank of {sl_name}', 'Test rows': n,
                          'Recall@1 %': round(100*hit/n, 1), '95% CI': wilson(hit, n), 'Cost US$ per 1,000 queries': round(usd_k, 3)})
rret = pd.DataFrame(rrows); rret.to_csv(RESULTS/'rq1_jira_api_rerank.csv', index=False); rret

## 8. Spend and summary

In [ ]:
spend = []
for f in RAW.glob('*.jsonl'):
    recs = [json.loads(l) for l in open(f)]
    spend.append({'File': f.name, 'Batches': len(recs), 'Input tokens': sum(r['input_tokens'] for r in recs),
                  'Output tokens': sum(r['output_tokens'] for r in recs), 'US$': round(sum(r['usd'] for r in recs), 3)})
spend = pd.DataFrame(spend); spend.to_csv(RESULTS/'rq1_api_spend.csv', index=False)
def mdt(d):
    try: return d.to_markdown(index=False)
    except Exception: return d.to_string()
lines = [f'# Step 07b summary - {datetime.datetime.now():%Y-%m-%d %H:%M}', '', '## Classification', mdt(api), '', '## Duplicate and cause re-ranking', mdt(rret),
         '', '## Spend', mdt(spend), f"\nTotal US$ {spend['US$'].sum():.2f}" if len(spend) else '']
(RESULTS/'step07b_summary.md').write_text('\n'.join(lines)); print('\n'.join(lines))